In [ ]:
# ============================================================
# NOTE 6 – COUNTRY ENERGY TRANSITION PROFILES
#
# RQ5:
# Các quốc gia có những kiểu chuyển dịch cơ cấu điện
# khác nhau như thế nào trong cùng một giai đoạn?
#
# Mục đích:
#   - Tiếp tục từ kết quả Note 5.
#   - Chọn giai đoạn 2012 → 2022.
#   - Kiểm tra coverage trước khi xây dựng đặc trưng.
#   - Chưa thực hiện clustering.
# ============================================================

import pandas as pd
from pathlib import Path

relative_path = Path("checkpoints/note05_event_validation.pkl")

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / relative_path).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Không tìm thấy file: {relative_path}"
    )
df_path = project_root / relative_path

note05_checkpoint = (
    project_root / "checkpoints/note05_event_validation.pkl"
)

note05_state = pd.read_pickle(note05_checkpoint)

df = note05_state["df"]
generation_sources = note05_state["generation_sources"]
system_sources = note05_state["system_sources"]

event_validation_final = note05_state[
    "event_validation_final"
]

# ------------------------------------------------------------
# Phạm vi phân tích
# ------------------------------------------------------------

start_year = 2012
end_year = 2022

country_transition = df[
    (df["Area type"] == "Country or economy") &
    (df["Year"].isin([start_year, end_year]))
].copy()

print("=" * 70)
print("NOTE 6 – COUNTRY ENERGY TRANSITION PROFILES")
print("=" * 70)

print()
print("RQ5:")
print("Các quốc gia có những kiểu chuyển dịch cơ cấu điện khác nhau như thế nào trong cùng một giai đoạn?")

print()
print("Giai đoạn phân tích:", start_year, "→", end_year)
print("Số dòng dữ liệu được chọn:", len(country_transition))

print()
print("Coverage theo năm:")

coverage = (
    country_transition
    .groupby("Year")["Area"]
    .nunique()
)

print(coverage.to_string())

print()
print("- Chưa clustering ở bước này.")
print("- Trước tiên phải xác định các biến có ý nghĩa để mô tả quá trình chuyển dịch của từng quốc gia.")
print("- Không dùng 2025 làm mốc so sánh quốc gia vì coverage quốc gia của năm này không đầy đủ.")

NOTE 6 – COUNTRY ENERGY TRANSITION PROFILES

RQ5:
Các quốc gia có những kiểu chuyển dịch cơ cấu điện khác nhau như thế nào trong cùng một giai đoạn?

Giai đoạn phân tích: 2012 → 2022
Số dòng dữ liệu được chọn: 6982

Coverage theo năm:
Year
2012    209
2022    209

- Chưa clustering ở bước này.
- Trước tiên phải xác định các biến có ý nghĩa để mô tả quá trình chuyển dịch của từng quốc gia.
- Không dùng 2025 làm mốc so sánh quốc gia vì coverage quốc gia của năm này không đầy đủ.


In [ ]:
# ============================================================
# KIỂM TRA COVERAGE CỦA CÁC BIẾN ỨNG VIÊN
#
# Những nguồn điện nào có dữ liệu đủ đầy để dùng làm
# đặc trưng mô tả quá trình chuyển dịch?
#
# Mục đích:
#   - Kiểm tra coverage 2012 và 2022.
#   - Chỉ giữ các quốc gia có đủ dữ liệu ở cả hai mốc.
#   - Tránh đưa feature thiếu dữ liệu nhiều vào bước phân nhóm.
# ============================================================

candidate_sources = [
    "Fossil",
    "Renewables",
    "Solar",
    "Wind",
    "Hydro",
    "Nuclear"
]

coverage_rows = []

for source in candidate_sources:

    source_data = country_transition[
        country_transition["Electricity source"] == source
    ]

    coverage_by_year = (
        source_data
        .groupby("Year")["Area"]
        .nunique()
    )

    countries_2012 = set(
        source_data.loc[
            source_data["Year"] == start_year,
            "Area"
        ]
    )

    countries_2022 = set(
        source_data.loc[
            source_data["Year"] == end_year,
            "Area"
        ]
    )

    common_countries = countries_2012 & countries_2022

    coverage_rows.append({
        "Source": source,
        "Countries_2012": len(countries_2012),
        "Countries_2022": len(countries_2022),
        "Complete_2012_2022": len(common_countries),
        "Missing_from_209": 209 - len(common_countries)
    })

source_coverage = pd.DataFrame(coverage_rows)

print("=" * 70)
print("SOURCE COVERAGE CHECK")
print("=" * 70)

print()
print("Coverage của các biến ứng viên:")
print(
    source_coverage.to_string(index=False)
)

print()
print("Tiêu chí:")
print("- Complete_2012_2022 = số quốc gia có dữ liệu ở cả 2 mốc.")
print("- Missing_from_209 = số quốc gia không đủ dữ liệu.")
print()
print("Chỉ những nguồn có coverage phù hợp mới được xem xét làm feature cho country profiling.")

SOURCE COVERAGE CHECK

Coverage của các biến ứng viên:
    Source  Countries_2012  Countries_2022  Complete_2012_2022  Missing_from_209
    Fossil             209             209                 209                 0
Renewables             209             209                 209                 0
     Solar             209             209                 209                 0
      Wind             201             206                 201                 8
     Hydro             205             206                 205                 4
   Nuclear             193             195                 193                16

Tiêu chí:
- Complete_2012_2022 = số quốc gia có dữ liệu ở cả 2 mốc.
- Missing_from_209 = số quốc gia không đủ dữ liệu.

Chỉ những nguồn có coverage phù hợp mới được xem xét làm feature cho country profiling.


In [3]:
# ============================================================
# XÂY DỰNG FEATURE CHUYỂN DỊCH 2012 → 2022
#
# Mức thay đổi tỷ trọng của từng nguồn điện giữa 2012
# và 2022 khác nhau như thế nào giữa các quốc gia?
#
# Mục đích:
#   - Tạo feature phản ánh TRANSITION thay vì chỉ quy mô.
#   - Δ share = Share_2022 − Share_2012.
#   - Không thay missing bằng 0.
# ============================================================

transition_sources = [
    "Fossil",
    "Renewables",
    "Solar",
    "Wind",
    "Hydro",
    "Nuclear"
]

# ------------------------------------------------------------
# Lấy dữ liệu tỷ trọng của các nguồn ứng viên
# ------------------------------------------------------------

transition_base = country_transition[
    country_transition["Electricity source"].isin(
        transition_sources
    )
][
    [
        "Area",
        "Year",
        "Electricity source",
        "Share of generation (%)"
    ]
].copy()

# ------------------------------------------------------------
# Pivot: mỗi quốc gia = một dòng
# ------------------------------------------------------------

transition_share = transition_base.pivot_table(
    index="Area",
    columns=["Year", "Electricity source"],
    values="Share of generation (%)",
    aggfunc="first"
)

# ------------------------------------------------------------
# Tính Δ share cho từng nguồn
# ------------------------------------------------------------

transition_features = pd.DataFrame(
    index=transition_share.index
)

for source in transition_sources:

    share_2012 = transition_share.get(
        (start_year, source)
    )

    share_2022 = transition_share.get(
        (end_year, source)
    )

    transition_features[
        f"Delta_{source}_share_pp"
    ] = share_2022 - share_2012

transition_features = transition_features.reset_index()

# ------------------------------------------------------------
# Kiểm tra kết quả
# ------------------------------------------------------------

print("=" * 70)
print("TRANSITION FEATURES 2012 → 2022")
print("=" * 70)

print()
print("Shape của bảng feature:")
print(transition_features.shape)

print()
print("Các feature được tạo:")
print(
    transition_features.columns.tolist()
)

print()
print("5 dòng đầu:")
print(
    transition_features
    .round(3)
    .head()
    .to_string(index=False)
)

print()
print("Missing values của từng feature:")
print(
    transition_features
    .isna()
    .sum()
    .to_string()
)

print()
print("Thống kê mô tả Δ share (percentage points):")
print(
    transition_features
    .drop(columns="Area")
    .describe()
    .round(3)
    .to_string()
)

TRANSITION FEATURES 2012 → 2022

Shape của bảng feature:
(209, 7)

Các feature được tạo:
['Area', 'Delta_Fossil_share_pp', 'Delta_Renewables_share_pp', 'Delta_Solar_share_pp', 'Delta_Wind_share_pp', 'Delta_Hydro_share_pp', 'Delta_Nuclear_share_pp']

5 dòng đầu:
          Area  Delta_Fossil_share_pp  Delta_Renewables_share_pp  Delta_Solar_share_pp  Delta_Wind_share_pp  Delta_Hydro_share_pp  Delta_Nuclear_share_pp
   Afghanistan                 -2.367                      2.367                 5.966                0.000                -3.599                     0.0
       Albania                  0.000                      0.000                 2.296                0.000                -2.296                     0.0
       Algeria                  0.140                     -0.140                 0.907                0.011                -1.058                     0.0
American Samoa                 -5.556                      5.556                 5.556                0.000               

In [4]:
# ==============================================================
# Quốc gia nào có mức thay đổi tỷ trọng Fossil, Renewables
# và Solar lớn nhất theo cả chiều tăng và chiều giảm?
#
# Mục đích:
# - Nhận diện các transition profile cực đoan.
# - Kiểm tra dữ liệu trước khi phân nhóm quốc gia.
# - Xác định liệu có các kiểu chuyển dịch rõ ràng hay không.
# ==============================================================

profile_features = [
    "Delta_Fossil_share_pp",
    "Delta_Renewables_share_pp",
    "Delta_Solar_share_pp",
    "Delta_Wind_share_pp",
    "Delta_Hydro_share_pp",
    "Delta_Nuclear_share_pp"
]

# --------------------------------------------------------------
# 1. Quốc gia có Fossil giảm mạnh nhất
# --------------------------------------------------------------
top_fossil_decline = (
    transition_features[
        ["Area", "Delta_Fossil_share_pp", "Delta_Renewables_share_pp",
         "Delta_Solar_share_pp", "Delta_Wind_share_pp"]
    ]
    .sort_values("Delta_Fossil_share_pp")
    .head(10)
)

print("=" * 70)
print("TOP 10 QUỐC GIA GIẢM TỶ TRỌNG FOSSIL MẠNH NHẤT")
print("=" * 70)
print(top_fossil_decline.to_string(index=False))


# --------------------------------------------------------------
# 2. Quốc gia có Renewables tăng mạnh nhất
# --------------------------------------------------------------
top_renewables_growth = (
    transition_features[
        ["Area", "Delta_Renewables_share_pp", "Delta_Fossil_share_pp",
         "Delta_Solar_share_pp", "Delta_Wind_share_pp"]
    ]
    .sort_values("Delta_Renewables_share_pp", ascending=False)
    .head(10)
)

print("\n" + "=" * 70)
print("TOP 10 QUỐC GIA TĂNG TỶ TRỌNG RENEWABLES MẠNH NHẤT")
print("=" * 70)
print(top_renewables_growth.to_string(index=False))


# --------------------------------------------------------------
# 3. Quốc gia có Solar tăng mạnh nhất
# --------------------------------------------------------------
top_solar_growth = (
    transition_features[
        ["Area", "Delta_Solar_share_pp", "Delta_Fossil_share_pp",
         "Delta_Renewables_share_pp"]
    ]
    .sort_values("Delta_Solar_share_pp", ascending=False)
    .head(10)
)

print("\n" + "=" * 70)
print("TOP 10 QUỐC GIA TĂNG TỶ TRỌNG SOLAR MẠNH NHẤT")
print("=" * 70)
print(top_solar_growth.to_string(index=False))


# --------------------------------------------------------------
# 4. Quốc gia có Fossil tăng mạnh nhất
# --------------------------------------------------------------
top_fossil_growth = (
    transition_features[
        ["Area", "Delta_Fossil_share_pp", "Delta_Renewables_share_pp",
         "Delta_Solar_share_pp"]
    ]
    .sort_values("Delta_Fossil_share_pp", ascending=False)
    .head(10)
)

print("\n" + "=" * 70)
print("TOP 10 QUỐC GIA TĂNG TỶ TRỌNG FOSSIL MẠNH NHẤT")
print("=" * 70)
print(top_fossil_growth.to_string(index=False))

TOP 10 QUỐC GIA GIẢM TỶ TRỌNG FOSSIL MẠNH NHẤT
                Area  Delta_Fossil_share_pp  Delta_Renewables_share_pp  Delta_Solar_share_pp  Delta_Wind_share_pp
          Luxembourg                -74.718                     74.718                21.691               22.721
  Cook Islands (the)                -50.000                     50.000                50.000                0.000
           Lithuania                -48.530                     48.530                 8.416               25.376
             Lebanon                -39.463                     39.463                28.833                0.229
            Djibouti                -35.000                     35.000                 0.000               35.000
             Liberia                -33.333                     33.333                 0.000                0.000
Palestine (State of)                -33.333                     33.333                33.333                0.000
             Denmark                -32.3

In [5]:
# ==============================================================
# Các thay đổi tỷ trọng cực lớn có phụ thuộc vào quy mô hệ thống điện của quốc gia hay không?
#
# Mục đích:
# - Bổ sung quy mô Total generation vào transition profile.
# - Phân biệt các quốc gia nhỏ có biến động tỷ trọng lớn với các quốc gia có hệ thống điện lớn.
# - Xác định có cần kiểm soát quy mô trước khi phân nhóm hay không.
# ==============================================================

# Lấy Total generation của từng quốc gia tại 2012 và 2022
generation_base = country_transition[
    (country_transition["Electricity source"] == "Total generation")
][
    ["Area", "Year", "Generation (TWh)"]
].copy()

generation_pivot = generation_base.pivot_table(
    index="Area",
    columns="Year",
    values="Generation (TWh)",
    aggfunc="first"
).reset_index()

generation_pivot = generation_pivot.rename(columns={
    start_year: "Generation_2012_TWh",
    end_year: "Generation_2022_TWh"
})

# Ghép với transition features
transition_profile = transition_features.merge(
    generation_pivot,
    on="Area",
    how="left"
)

print("=" * 70)
print("TRANSITION PROFILE + QUY MÔ HỆ THỐNG ĐIỆN")
print("=" * 70)

print("\nShape:", transition_profile.shape)

print("\n5 dòng đầu:")
print(
    transition_profile[
        [
            "Area",
            "Generation_2012_TWh",
            "Generation_2022_TWh",
            "Delta_Fossil_share_pp",
            "Delta_Renewables_share_pp",
            "Delta_Solar_share_pp"
        ]
    ].head().to_string(index=False)
)

# --------------------------------------------------------------
# So sánh các nhóm extreme với quy mô hệ thống điện
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("10 QUỐC GIA GIẢM FOSSIL MẠNH NHẤT + QUY MÔ HỆ THỐNG")
print("=" * 70)

extreme_decline = (
    transition_profile[
        [
            "Area",
            "Generation_2012_TWh",
            "Generation_2022_TWh",
            "Delta_Fossil_share_pp",
            "Delta_Renewables_share_pp"
        ]
    ]
    .sort_values("Delta_Fossil_share_pp")
    .head(10)
)

print(extreme_decline.to_string(index=False))


print("\n" + "=" * 70)
print("10 QUỐC GIA TĂNG FOSSIL MẠNH NHẤT + QUY MÔ HỆ THỐNG")
print("=" * 70)

extreme_growth = (
    transition_profile[
        [
            "Area",
            "Generation_2012_TWh",
            "Generation_2022_TWh",
            "Delta_Fossil_share_pp",
            "Delta_Renewables_share_pp"
        ]
    ]
    .sort_values("Delta_Fossil_share_pp", ascending=False)
    .head(10)
)

print(extreme_growth.to_string(index=False))


# --------------------------------------------------------------
# Phân vị quy mô hệ thống điện
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("PHÂN PHỐI QUY MÔ HỆ THỐNG ĐIỆN 2022")
print("=" * 70)

print(
    transition_profile["Generation_2022_TWh"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
    .round(3)
)

TRANSITION PROFILE + QUY MÔ HỆ THỐNG ĐIỆN

Shape: (209, 9)

5 dòng đầu:
          Area  Generation_2012_TWh  Generation_2022_TWh  Delta_Fossil_share_pp  Delta_Renewables_share_pp  Delta_Solar_share_pp
   Afghanistan                 0.88                 0.96                 -2.367                      2.367                 5.966
       Albania                 4.72                 6.97                  0.000                      0.000                 2.296
       Algeria                57.40                91.47                  0.140                     -0.140                 0.907
American Samoa                 0.16                 0.18                 -5.556                      5.556                 5.556
        Angola                 7.72                17.31                 -4.579                      4.579                 2.123

10 QUỐC GIA GIẢM FOSSIL MẠNH NHẤT + QUY MÔ HỆ THỐNG
                Area  Generation_2012_TWh  Generation_2022_TWh  Delta_Fossil_share_pp  Delta_Renewabl

In [8]:
# ==============================================================
# Có thể mô tả mỗi quốc gia bằng hướng chuyển dịch, cường độ chuyển dịch và nguồn điện đóng góp chính hay không?
#
# Mục đích:
# - Xây dựng transition profile có ý nghĩa diễn giải.
# - Không phụ thuộc vào clustering ở bước này.
# - Chuẩn bị feature cho bước phân nhóm nếu EDA cho thấy phù hợp.
# ==============================================================

profile = transition_profile.copy()

# --------------------------------------------------------------
# 1. Hướng chuyển dịch Fossil
# --------------------------------------------------------------
import numpy as np

profile["Transition_direction"] = np.select(
    [
        profile["Delta_Fossil_share_pp"] <= -5,
        profile["Delta_Fossil_share_pp"] >= 5
    ],
    [
        "Fossil decline",
        "Fossil growth"
    ],
    default="Stable / moderate"
)


# --------------------------------------------------------------
# 2. Cường độ chuyển dịch
# Dùng độ lớn thay đổi Fossil share làm chỉ báo chính.
# --------------------------------------------------------------
profile["Transition_intensity_pp"] = (
    profile["Delta_Fossil_share_pp"].abs()
)


# --------------------------------------------------------------
# 3. Nguồn tái tạo nổi bật
# So sánh mức tăng Solar / Wind / Hydro.
# Nuclear được giữ riêng vì không thuộc Renewables.
# --------------------------------------------------------------
renewable_delta_cols = [
    "Delta_Solar_share_pp",
    "Delta_Wind_share_pp",
    "Delta_Hydro_share_pp"
]

profile["Dominant_renewable_change"] = (
    profile[renewable_delta_cols]
    .abs()
    .idxmax(axis=1)
    .str.replace("Delta_", "", regex=False)
    .str.replace("_share_pp", "", regex=False)
)


# --------------------------------------------------------------
# 4. Mức thay đổi Solar
# --------------------------------------------------------------
profile["Solar_transition_strength"] = (
    profile["Delta_Solar_share_pp"].abs()
)


# --------------------------------------------------------------
# 5. Tóm tắt các profile
# --------------------------------------------------------------
print("=" * 70)
print("TRANSITION PROFILE SUMMARY")
print("=" * 70)

print("\nShape:", profile.shape)

print("\nPhân bố hướng chuyển dịch:")
print(
    profile["Transition_direction"]
    .value_counts()
)

print("\nNguồn renewable có thay đổi nổi bật nhất:")
print(
    profile["Dominant_renewable_change"]
    .value_counts(dropna=False)
)

print("\nThống kê cường độ chuyển dịch:")
print(
    profile["Transition_intensity_pp"]
    .describe()
    .round(3)
)


# --------------------------------------------------------------
# 6. Những quốc gia có transition intensity lớn nhất
# --------------------------------------------------------------
print("\n" + "=" * 70)
print("TOP 15 QUỐC GIA CÓ CƯỜNG ĐỘ CHUYỂN DỊCH LỚN NHẤT")
print("=" * 70)

top_transition = (
    profile[
        [
            "Area",
            "Generation_2022_TWh",
            "Delta_Fossil_share_pp",
            "Delta_Renewables_share_pp",
            "Delta_Solar_share_pp",
            "Delta_Wind_share_pp",
            "Delta_Hydro_share_pp",
            "Delta_Nuclear_share_pp",
            "Transition_direction",
            "Transition_intensity_pp",
            "Dominant_renewable_change"
        ]
    ]
    .sort_values("Transition_intensity_pp", ascending=False)
    .head(15)
)

print(top_transition.to_string(index=False))

TRANSITION PROFILE SUMMARY

Shape: (209, 13)

Phân bố hướng chuyển dịch:
Transition_direction
Stable / moderate    95
Fossil decline       94
Fossil growth        20
Name: count, dtype: int64

Nguồn renewable có thay đổi nổi bật nhất:
Dominant_renewable_change
Solar    91
Hydro    80
Wind     38
Name: count, dtype: int64

Thống kê cường độ chuyển dịch:
count    209.000
mean       9.804
std       11.122
min        0.000
25%        1.823
50%        6.667
75%       14.132
max       74.718
Name: Transition_intensity_pp, dtype: float64

TOP 15 QUỐC GIA CÓ CƯỜNG ĐỘ CHUYỂN DỊCH LỚN NHẤT
                       Area  Generation_2022_TWh  Delta_Fossil_share_pp  Delta_Renewables_share_pp  Delta_Solar_share_pp  Delta_Wind_share_pp  Delta_Hydro_share_pp  Delta_Nuclear_share_pp Transition_direction  Transition_intensity_pp Dominant_renewable_change
                 Luxembourg                 1.21                -74.718                     74.718                21.691               22.721            

In [10]:
# ==============================================================
# Mối quan hệ giữa thay đổi Fossil và thay đổi Renewables có hình thành các nhóm chuyển dịch tự nhiên hay không?
#
# Mục đích:
# - Kiểm tra cấu trúc transition profile trước clustering.
# - Xác định các kiểu:
#   1. Fossil giảm / Renewable tăng
#   2. Fossil tăng / Renewable giảm
#   3. Cả hai thay đổi ít
# - Không thực hiện clustering ở bước này.
# ==============================================================

transition_relation = profile[
    [
        "Area",
        "Generation_2022_TWh",
        "Delta_Fossil_share_pp",
        "Delta_Renewables_share_pp",
        "Delta_Solar_share_pp",
        "Delta_Wind_share_pp",
        "Delta_Hydro_share_pp",
        "Transition_direction",
        "Transition_intensity_pp"
    ]
].copy()

print("=" * 70)
print("FOSSIL ↔ RENEWABLE TRANSITION RELATION")
print("=" * 70)

print("\nSố quốc gia:", len(transition_relation))

print("\nCorrelation giữa thay đổi Fossil và Renewables:")

print(
    transition_relation[
        [
            "Delta_Fossil_share_pp",
            "Delta_Renewables_share_pp"
        ]
    ]
    .corr()
    .round(3)
)

print("\n" + "=" * 70)
print("CÁC QUỐC GIA: FOSSIL GIẢM + RENEWABLES TĂNG")
print("=" * 70)

decline_transition = transition_relation[
    (transition_relation["Delta_Fossil_share_pp"] <= -5) &
    (transition_relation["Delta_Renewables_share_pp"] >= 5)
].copy()

print("Số quốc gia:", len(decline_transition))

print(
    decline_transition[
        [
            "Area",
            "Generation_2022_TWh",
            "Delta_Fossil_share_pp",
            "Delta_Renewables_share_pp",
            "Delta_Solar_share_pp",
            "Delta_Wind_share_pp",
            "Delta_Hydro_share_pp"
        ]
    ]
    .sort_values("Delta_Fossil_share_pp")
    .head(15)
    .to_string(index=False)
)


print("\n" + "=" * 70)
print("CÁC QUỐC GIA: FOSSIL TĂNG + RENEWABLES GIẢM")
print("=" * 70)

reverse_transition = transition_relation[
    (transition_relation["Delta_Fossil_share_pp"] >= 5) &
    (transition_relation["Delta_Renewables_share_pp"] <= -5)
].copy()

print("Số quốc gia:", len(reverse_transition))

print(
    reverse_transition[
        [
            "Area",
            "Generation_2022_TWh",
            "Delta_Fossil_share_pp",
            "Delta_Renewables_share_pp",
            "Delta_Solar_share_pp",
            "Delta_Wind_share_pp",
            "Delta_Hydro_share_pp"
        ]
    ]
    .sort_values("Delta_Fossil_share_pp", ascending=False)
    .head(15)
    .to_string(index=False)
)


print("\n" + "=" * 70)
print("CÁC QUỐC GIA THAY ĐỔI CƠ CẤU NHỎ")
print("=" * 70)

stable_transition = transition_relation[
    transition_relation["Transition_intensity_pp"] < 5
].copy()

print("Số quốc gia:", len(stable_transition))

print(
    stable_transition[
        [
            "Area",
            "Generation_2022_TWh",
            "Delta_Fossil_share_pp",
            "Delta_Renewables_share_pp",
            "Transition_intensity_pp"
        ]
    ]
    .sort_values("Transition_intensity_pp")
    .head(15)
    .to_string(index=False)
)

FOSSIL ↔ RENEWABLE TRANSITION RELATION

Số quốc gia: 209

Correlation giữa thay đổi Fossil và Renewables:
                           Delta_Fossil_share_pp  Delta_Renewables_share_pp
Delta_Fossil_share_pp                      1.000                     -0.986
Delta_Renewables_share_pp                 -0.986                      1.000

CÁC QUỐC GIA: FOSSIL GIẢM + RENEWABLES TĂNG
Số quốc gia: 88
                Area  Generation_2022_TWh  Delta_Fossil_share_pp  Delta_Renewables_share_pp  Delta_Solar_share_pp  Delta_Wind_share_pp  Delta_Hydro_share_pp
          Luxembourg                 1.21                -74.718                     74.718                21.691               22.721                 1.336
  Cook Islands (the)                 0.04                -50.000                     50.000                50.000                0.000                 0.000
           Lithuania                 4.04                -48.530                     48.530                 8.416               25.376

In [11]:
# ==============================================================
# Các transition profile có phụ thuộc mạnh vào quy mô hệ thống điện của quốc gia hay không?
#
# Mục đích:
# - Kiểm tra robustness của kết quả transition profile.
# - Không để các quốc gia có hệ thống điện cực nhỏ chi phối cách diễn giải.
# - So sánh Fossil decline / stable / Fossil growth giữa các nhóm quy mô.
# ==============================================================

size_profile = profile.copy()

# Chia quy mô hệ thống điện 2022 thành 4 nhóm theo quartile
size_profile["Generation_size_group"] = pd.qcut(
    size_profile["Generation_2022_TWh"],
    q=4,
    labels=[
        "Smallest 25%",
        "Small-Medium",
        "Medium-Large",
        "Largest 25%"
    ],
    duplicates="drop"
)

# --------------------------------------------------------------
# 1. Số quốc gia trong từng nhóm quy mô
# --------------------------------------------------------------
print("=" * 70)
print("SỐ QUỐC GIA THEO QUY MÔ HỆ THỐNG ĐIỆN")
print("=" * 70)

print(
    size_profile["Generation_size_group"]
    .value_counts()
    .sort_index()
)


# --------------------------------------------------------------
# 2. Transition direction trong từng nhóm quy mô
# --------------------------------------------------------------
print("\n" + "=" * 70)
print("TRANSITION DIRECTION × QUY MÔ HỆ THỐNG")
print("=" * 70)

transition_size_table = pd.crosstab(
    size_profile["Generation_size_group"],
    size_profile["Transition_direction"]
)

print(transition_size_table)


# --------------------------------------------------------------
# 3. Tỷ lệ (%) trong từng nhóm quy mô
# --------------------------------------------------------------
print("\n" + "=" * 70)
print("TỶ LỆ TRANSITION PROFILE (%) THEO QUY MÔ")
print("=" * 70)

transition_size_pct = pd.crosstab(
    size_profile["Generation_size_group"],
    size_profile["Transition_direction"],
    normalize="index"
) * 100

print(
    transition_size_pct
    .round(2)
)


# --------------------------------------------------------------
# 4. Cường độ chuyển dịch trung bình
# --------------------------------------------------------------
print("\n" + "=" * 70)
print("CƯỜNG ĐỘ CHUYỂN DỊCH THEO QUY MÔ")
print("=" * 70)

intensity_by_size = (
    size_profile
    .groupby("Generation_size_group", observed=True)
    ["Transition_intensity_pp"]
    .agg(["count", "mean", "median", "max"])
    .round(3)
)

print(intensity_by_size)

SỐ QUỐC GIA THEO QUY MÔ HỆ THỐNG ĐIỆN
Generation_size_group
Smallest 25%    53
Small-Medium    52
Medium-Large    52
Largest 25%     52
Name: count, dtype: int64

TRANSITION DIRECTION × QUY MÔ HỆ THỐNG
Transition_direction   Fossil decline  Fossil growth  Stable / moderate
Generation_size_group                                                  
Smallest 25%                       21              5                 27
Small-Medium                       26              6                 20
Medium-Large                       22              7                 23
Largest 25%                        25              2                 25

TỶ LỆ TRANSITION PROFILE (%) THEO QUY MÔ
Transition_direction   Fossil decline  Fossil growth  Stable / moderate
Generation_size_group                                                  
Smallest 25%                    39.62           9.43              50.94
Small-Medium                    50.00          11.54              38.46
Medium-Large                    42.3

In [12]:
# ============================================================
# PHÂN RÃ NHÓM FOSSIL DECLINE
#
# Trong nhóm Fossil decline, nguồn tái tạo nào là động lực chính của sự thay đổi: Solar, Wind hay Hydro?
#
# Mục đích: Xác định các transition profile có ý nghĩa về mặt diễn giải
# trước khi quyết định có cần clustering hay không.
# ============================================================

decline_profile = profile[
    profile["Transition_direction"] == "Fossil decline"
].copy()

renewable_cols = [
    "Delta_Solar_share_pp",
    "Delta_Wind_share_pp",
    "Delta_Hydro_share_pp"
]

# Nguồn tái tạo có mức tăng tỷ trọng lớn nhất
decline_profile["Dominant_renewable"] = (
    decline_profile[renewable_cols]
    .idxmax(axis=1)
    .str.replace("Delta_", "", regex=False)
    .str.replace("_share_pp", "", regex=False)
)

print("=" * 70)
print("SỐ QUỐC GIA TRONG NHÓM FOSSIL DECLINE")
print("=" * 70)
print(f"Tổng số quốc gia: {len(decline_profile)}")

print("\n" + "=" * 70)
print("NGUỒN TÁI TẠO DẪN DẮT")
print("=" * 70)

dominant_counts = (
    decline_profile["Dominant_renewable"]
    .value_counts()
)

dominant_pct = (
    decline_profile["Dominant_renewable"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

dominant_table = pd.DataFrame({
    "Count": dominant_counts,
    "Percentage": dominant_pct
})

print(dominant_table)

print("\n" + "=" * 70)
print("MỨC THAY ĐỔI TỶ TRỌNG TRUNG BÌNH")
print("=" * 70)

print(
    decline_profile[
        [
            "Delta_Solar_share_pp",
            "Delta_Wind_share_pp",
            "Delta_Hydro_share_pp"
        ]
    ]
    .agg(["mean", "median", "min", "max"])
    .round(3)
)

print("\n" + "=" * 70)
print("10 QUỐC GIA CÓ MỨC TĂNG SOLAR CAO NHẤT")
print("=" * 70)

print(
    decline_profile[
        ["Area", "Delta_Solar_share_pp",
         "Delta_Wind_share_pp", "Delta_Hydro_share_pp"]
    ]
    .sort_values("Delta_Solar_share_pp", ascending=False)
    .head(10)
    .round(3)
)

print("\n" + "=" * 70)
print("10 QUỐC GIA CÓ MỨC TĂNG WIND CAO NHẤT")
print("=" * 70)

print(
    decline_profile[
        ["Area", "Delta_Solar_share_pp",
         "Delta_Wind_share_pp", "Delta_Hydro_share_pp"]
    ]
    .sort_values("Delta_Wind_share_pp", ascending=False)
    .head(10)
    .round(3)
)

print("\n" + "=" * 70)
print("10 QUỐC GIA CÓ MỨC TĂNG HYDRO CAO NHẤT")
print("=" * 70)

print(
    decline_profile[
        ["Area", "Delta_Solar_share_pp",
         "Delta_Wind_share_pp", "Delta_Hydro_share_pp"]
    ]
    .sort_values("Delta_Hydro_share_pp", ascending=False)
    .head(10)
    .round(3)
)

SỐ QUỐC GIA TRONG NHÓM FOSSIL DECLINE
Tổng số quốc gia: 94

NGUỒN TÁI TẠO DẪN DẮT
                    Count  Percentage
Dominant_renewable                   
Solar                  47       50.00
Wind                   27       28.72
Hydro                  20       21.28

MỨC THAY ĐỔI TỶ TRỌNG TRUNG BÌNH
        Delta_Solar_share_pp  Delta_Wind_share_pp  Delta_Hydro_share_pp
mean                   7.011                5.051                 1.712
median                 4.946                1.863                 0.000
min                    0.000               -1.789               -28.685
max                   50.000               35.000                33.333

10 QUỐC GIA CÓ MỨC TĂNG SOLAR CAO NHẤT
                     Area  Delta_Solar_share_pp  Delta_Wind_share_pp  \
42     Cook Islands (the)                50.000                0.000   
143  Palestine (State of)                33.333                0.000   
105               Lebanon                28.833                0.229   
99    

In [13]:
# ============================================================
# SO SÁNH 3 KIỂU CHUYỂN DỊCH
#
# Solar-dominant, Wind-dominant và Hydro-dominant có thực sự
# khác nhau về cường độ chuyển dịch và mức thay đổi cơ cấu
# hay chỉ khác nhau ở nguồn tái tạo tăng mạnh nhất?
#
# Mục đích:Kiểm tra tính hợp lý của việc xây dựng transition profiles
# trước khi quyết định có cần clustering hay không.
# ============================================================

profile_compare = decline_profile[
    [
        "Area",
        "Dominant_renewable",
        "Transition_intensity_pp",
        "Delta_Fossil_share_pp",
        "Delta_Renewables_share_pp",
        "Delta_Solar_share_pp",
        "Delta_Wind_share_pp",
        "Delta_Hydro_share_pp",
        "Delta_Nuclear_share_pp",
        "Generation_2022_TWh"
    ]
].copy()

print("=" * 70)
print("SỐ QUỐC GIA THEO TRANSITION PROFILE")
print("=" * 70)

print(
    profile_compare["Dominant_renewable"]
    .value_counts()
)

print("\n" + "=" * 70)
print("THỐNG KÊ CHUYỂN DỊCH THEO PROFILE")
print("=" * 70)

profile_summary = (
    profile_compare
    .groupby("Dominant_renewable")
    .agg(
        Countries=("Area", "count"),
        Mean_intensity_pp=("Transition_intensity_pp", "mean"),
        Median_intensity_pp=("Transition_intensity_pp", "median"),
        Mean_fossil_change_pp=("Delta_Fossil_share_pp", "mean"),
        Mean_renewables_change_pp=("Delta_Renewables_share_pp", "mean"),
        Mean_solar_change_pp=("Delta_Solar_share_pp", "mean"),
        Mean_wind_change_pp=("Delta_Wind_share_pp", "mean"),
        Mean_hydro_change_pp=("Delta_Hydro_share_pp", "mean"),
        Mean_nuclear_change_pp=("Delta_Nuclear_share_pp", "mean"),
        Median_generation_2022_TWh=("Generation_2022_TWh", "median")
    )
    .round(3)
)

print(profile_summary)

print("\n" + "=" * 70)
print("PHÂN BỐ CƯỜNG ĐỘ CHUYỂN DỊCH")
print("=" * 70)

intensity_summary = (
    profile_compare
    .groupby("Dominant_renewable")["Transition_intensity_pp"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Q1=lambda x: x.quantile(0.25),
        Q3=lambda x: x.quantile(0.75),
        Max="max"
    )
    .round(3)
)

print(intensity_summary)

SỐ QUỐC GIA THEO TRANSITION PROFILE
Dominant_renewable
Solar    47
Wind     27
Hydro    20
Name: count, dtype: int64

THỐNG KÊ CHUYỂN DỊCH THEO PROFILE
                    Countries  Mean_intensity_pp  Median_intensity_pp  \
Dominant_renewable                                                      
Hydro                      20             16.182               15.252   
Solar                      47             14.598               11.739   
Wind                       27             19.457               14.682   

                    Mean_fossil_change_pp  Mean_renewables_change_pp  \
Dominant_renewable                                                     
Hydro                             -16.182                     15.031   
Solar                             -14.598                     13.846   
Wind                              -19.457                     19.913   

                    Mean_solar_change_pp  Mean_wind_change_pp  \
Dominant_renewable                                      

In [14]:
# ============================================================
# PROFILE × QUY MÔ HỆ THỐNG
#
# Solar/Wind/Hydro-dominant có xuất hiện ở nhiều quy mô hệ thống khác nhau hay chỉ tập trung ở một nhóm quy mô?
#
# Mục đích: Kiểm tra transition profile có mang tính cơ cấu hay chỉ là hệ quả của quy mô hệ thống điện.
# ============================================================

profile_size = profile_compare.copy()

# Ghép nhóm quy mô đã tạo ở Step 6.8
profile_size = profile_size.merge(
    size_profile[
        ["Area", "Generation_size_group"]
    ],
    on="Area",
    how="left"
)

print("=" * 70)
print("SỐ QUỐC GIA THEO TRANSITION PROFILE × QUY MÔ")
print("=" * 70)

profile_size_count = pd.crosstab(
    profile_size["Generation_size_group"],
    profile_size["Dominant_renewable"]
)

print(profile_size_count)

print("\n" + "=" * 70)
print("TỶ LỆ PROFILE TRONG TỪNG NHÓM QUY MÔ (%)")
print("=" * 70)

profile_size_pct = (
    pd.crosstab(
        profile_size["Generation_size_group"],
        profile_size["Dominant_renewable"],
        normalize="index"
    )
    .mul(100)
    .round(2)
)

print(profile_size_pct)

print("\n" + "=" * 70)
print("MEDIAN GENERATION THEO PROFILE × QUY MÔ")
print("=" * 70)

profile_size_generation = (
    profile_size
    .groupby(
        ["Generation_size_group", "Dominant_renewable"],
        observed=True
    )["Generation_2022_TWh"]
    .agg(["count", "median"])
    .round(3)
)

print(profile_size_generation)

SỐ QUỐC GIA THEO TRANSITION PROFILE × QUY MÔ
Dominant_renewable     Hydro  Solar  Wind
Generation_size_group                    
Smallest 25%               5     14     2
Small-Medium               7     12     7
Medium-Large               5      8     9
Largest 25%                3     13     9

TỶ LỆ PROFILE TRONG TỪNG NHÓM QUY MÔ (%)
Dominant_renewable     Hydro  Solar   Wind
Generation_size_group                     
Smallest 25%           23.81  66.67   9.52
Small-Medium           26.92  46.15  26.92
Medium-Large           22.73  36.36  40.91
Largest 25%            12.00  52.00  36.00

MEDIAN GENERATION THEO PROFILE × QUY MÔ
                                          count   median
Generation_size_group Dominant_renewable                
Smallest 25%          Hydro                   5    0.390
                      Solar                  14    0.270
                      Wind                    2    0.340
Small-Medium          Hydro                   7    5.750
                    

Trong giai đoạn 2012–2022, các quốc gia thể hiện ba trạng thái chuyển dịch chính: giảm tỷ trọng fossil, tăng tỷ trọng fossil và ổn định/tương đối ổn định. Trong nhóm giảm fossil, có thể nhận diện ba kiểu chuyển dịch dựa trên nguồn tái tạo dẫn dắt: Solar-dominant (50.00%), Wind-dominant (28.72%) và Hydro-dominant (21.28%). Các profile này xuất hiện ở nhiều quy mô hệ thống điện, mặc dù cơ cấu profile thay đổi theo quy mô. Điều này cho thấy quá trình chuyển dịch năng lượng không diễn ra theo một mô hình duy nhất giữa các quốc gia.

In [15]:

from pathlib import Path
import pickle

checkpoint_path = project_root / "checkpoints/note06_country_transition_profiles.pkl"

note06_objects = {
    "country_transition": country_transition,
    "transition_features": transition_features,
    "transition_profile": transition_profile,
    "profile": profile,
    "size_profile": size_profile,
    "transition_size_table": transition_size_table,
    "intensity_by_size": intensity_by_size,
    "decline_profile": decline_profile,
    "profile_compare": profile_compare,
    "profile_size": profile_size,
    "profile_size_count": profile_size_count,
    "profile_size_pct": profile_size_pct,
    "profile_size_generation": profile_size_generation
}

with open(checkpoint_path, "wb") as f:
    pickle.dump(note06_objects, f)

print("=" * 70)
print("NOTE 6 — CHECKPOINT SAVED")
print("=" * 70)
print(f"File: {checkpoint_path}")
print(f"Số object đã lưu: {len(note06_objects)}")
print()
print("RQ5 đã hoàn thành:")
print("✓ Phân loại hướng chuyển dịch")
print("✓ Kiểm tra theo quy mô hệ thống")
print("✓ Xác định Solar / Wind / Hydro transition profiles")
print("✓ Kiểm tra profile theo quy mô")

NOTE 6 — CHECKPOINT SAVED
File: d:\phan tich data\Ember_Electricity\checkpoints\note06_country_transition_profiles.pkl
Số object đã lưu: 13

RQ5 đã hoàn thành:
✓ Phân loại hướng chuyển dịch
✓ Kiểm tra theo quy mô hệ thống
✓ Xác định Solar / Wind / Hydro transition profiles
✓ Kiểm tra profile theo quy mô
